# 일단 ai 헬멧 감지 

In [1]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 주피터 노트북용 비동기 패치 (에러 방지)
nest_asyncio.apply()

# 오라클 클라이언트 초기화
try:
    oracledb.init_oracle_client()
except Exception as e:
    pass

# FastAPI 앱 생성
app = FastAPI()

# YOLO 모델 로딩
model_path = "data/best (sDUDU).pt"

model = None
try:
    if os.path.exists(model_path):
        model = YOLO(model_path)
        print(f"ai 모델 로딩 성공({model+path})")
        print(f"감지 가능목록(names): {model.names}")
    else:
        print(f"파일이 없음:{os.path.abspath(model_path)}")
except Exception as e:
    print(f"모델 로딩 중 에러: {e}")

# db 연결 함수
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"db접속 에러 : {e}")
        return None

# 메인 기능 : 자바가 요청 -> 카메라 켜고 감지 -> db 저장
@app.get("/helmet-check")
def helmet_check(kickboard_id: str):
    print(f"요청도착 킥보드id : {kickboard_id}")

    # ai 감지 로직
    helmet_status = "미착용"
    is_helmet_detected = False
    detected_objects = []

    if model:
        # 동영상 파일 경로
        video_path = "data/야간1(헬멧o).mp4"
        print(f"동영상({video_path})을 분석합니다.")
        
        if os.path.exists(video_path):
            # stream=True : 한번에 처리 x, 프레임별로 처리
            # max_det=1 : 한 프레임당 1개만 감지
            # vid_stride=30 : 모든 프레임x 30프레임마다 1번씩 검사
            results = model.predict(source=video_path, save=True, conf=0.5, vid_stride=30)

            # 결과 분석
            for result in results:
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    class_name = model.names[cls_id]

                    # 중복 제거해 리스트에 담기
                    if class_name not in detected_objects:
                        detected_objects.append(class_name)
                        print(f"ai가본것:{class_name}")

                    # 헬멧 감지 여부 체크
                    if 'helmet' in class_name.lower():
                        is_helmet_detected = True
                        helmet_status = "착용"
                        # 헬멧을 찾으면 더 검사 X 
                        break
                if is_helmet_detected: break
        else:
            print(f"영상 파일이 없음:{video_path}")
    else:
        print(f"모델이 없음")

모델 로딩 중 에러: name 'path' is not defined


In [2]:
import os  # ★ 이 친구가 없으면 path 관련 에러가 납니다!
from ultralytics import YOLO

# 경로 설정 (아까 만든 data 폴더)
model_path = "data/best.pt"

# 디버깅용: 현재 폴더 위치와 파일이 진짜 있는지 확인
print(f"📂 현재 작업 위치: {os.getcwd()}")

if os.path.exists(model_path):  # ★ 여기에 os.path 라고 정확히 써야 합니다!
    try:
        model = YOLO(model_path)
        print(f"✅ 모델 로딩 성공! ({model_path})")
        print(f"📋 감지 목록: {model.names}")
    except Exception as e:
        print(f"💥 모델 파일은 있는데 로딩 실패: {e}")
else:
    # 파일이 없을 때 절대경로를 보여줌 (찾기 쉽게)
    print(f"💥 파일을 못 찾겠어요! 여기 있는지 확인해보세요: {os.path.abspath(model_path)}")
    model = None

📂 현재 작업 위치: C:\Users\smhrd\GitHub\RealDuDu\jupiter_python
💥 파일을 못 찾겠어요! 여기 있는지 확인해보세요: C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\data\best.pt


In [3]:
import os

print("📂 'data' 폴더 안에 있는 실제 파일 목록:")
print("-" * 30)

try:
    files = os.listdir("data") # data 폴더 안을 들여다봅니다.
    for f in files:
        print(f"👉 발견된 파일: {f}")
        
    if not files:
        print("텅 비어있는데요? 😅")
        
except Exception as e:
    print(f"💥 에러! 'data'라는 폴더 자체가 없는 것 같아요. (현재 위치에 있는 폴더들: {os.listdir('.')})")

📂 'data' 폴더 안에 있는 실제 파일 목록:
------------------------------
👉 발견된 파일: 299823_tiny.mp4
👉 발견된 파일: 39183-421020269_tiny.mp4
👉 발견된 파일: best (sDUDU).pt
👉 발견된 파일: minha.mp4
👉 발견된 파일: minha2.mp4
👉 발견된 파일: results.csv
👉 발견된 파일: results.png
👉 발견된 파일: test1.mp4
👉 발견된 파일: testvideo.mp4
👉 발견된 파일: testvideo1.mp4
👉 발견된 파일: testvideo_demo.mp4
👉 발견된 파일: 야간1(노헬멧).mp4
👉 발견된 파일: 야간1(헬멧o).mp4
👉 발견된 파일: 야간2(헬멧o).mp4
